# Concurrent Four-Node BB84 and E91 Key Generation

**Audience:** professors, researchers, and students evaluating SimYuj's physical-component and protocol-composition capabilities.

**Prerequisites:** basic BB84, entanglement, CHSH, and QKD post-processing concepts. Run this notebook from the repository or a descendant directory using the project virtual environment.

**Learning goals:** build one four-node physical network, run BB84 and E91 concurrently, inspect loss and detector statistics, and verify two independently generated final keys.


## Outline

1. Load the configurable scenario.
2. Inspect the four-node physical topology.
3. Run both protocols on one event timeline.
4. Inspect BB84 and E91 post-processing.
5. Confirm both final keys and explore customization.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "simyuj").exists() and (candidate / "tutorials").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not find the SimYuj repository root")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tutorials.four_node_qkd import (
    FourNodeQKDConfig,
    run_four_node_qkd_trial,
    summarize_four_node_trial,
)
from tutorials.four_node_qkd.trial import build_four_node_qkd_trial
from dataclasses import replace

base = FourNodeQKDConfig()

config = replace(
    base,
    # BB84: 30,000 → 60,000 slots
    bb84_source=replace(
        base.bb84_source,
        num_slots=60_000,
    ),

    # E91: 50,000 → 100,000 pair attempts
    e91_source=replace(
        base.e91_source,
        num_slots=100_000,
    ),
)
print("Repository root:", REPO_ROOT)
print("Master seed:", config.master_seed)


## Physical topology

A sends BB84 photons to B. C sends the two members of each `psi-` pair to B and D. B contains separate BB84 and E91 receiver stacks. Public discussion uses directed classical fibers with propagation delay. No source, detector, or channel stub is used.

```text
A -- BB84 --> B

             +--> B
C -- pairs --|
             +--> D       E91 key: B <--> D
```


In [ ]:
preview = build_four_node_qkd_trial(config)
print("Nodes:", tuple(preview.network.nodes))
print("Quantum links:", tuple(preview.network.quantum_links))
print("Classical links:", tuple(preview.network.classical_links))
print("B devices:", tuple(preview.network.nodes["B"].devices))
print("Quantum component types:", {
    name: type(channel).__name__
    for name, channel in preview.quantum_channels.items()
})


## Run both protocols concurrently

Both source agents start at tick zero. The call below runs until every quantum, detector, agent, Cascade, verification, and privacy-amplification event has been consumed. Full key material is deliberately omitted from the returned report.


In [ ]:
result = run_four_node_qkd_trial(config)
print(summarize_four_node_trial(result))


## Physical-channel evidence

The counts below come from the real channel components. Different losses on C-B and C-D are expected because each arm has its own deterministic random stream.


In [ ]:
for name, stats in result["channels"]["quantum"].items():
    print(
        f"{name:12s} received={stats['received']:6d} "
        f"delivered={stats['delivered']:6d} lost={stats['lost']:6d}"
    )
print("Quantum frames overlapped:", result["concurrency"]["quantum_frames_overlapped"])
print("BB84 post-processing began during E91 transmission:",
      result["concurrency"]["bb84_postprocessing_started_before_e91_frame_end"])


## Complete BB84 workflow on A-B

The existing physical BB84 agents perform timing-based sifting, public QBER sampling, interactive Cascade, Toeplitz verification, and Toeplitz privacy amplification over A-B classical fibers.


In [ ]:
bb84 = result["bb84"]
for key in (
    "prepared_photons", "successful_detections", "sifted_bits",
    "estimated_qber", "cascade_parity_requests",
    "cascade_corrections", "verification_accepted",
    "final_key_length", "final_keys_equal", "protocol_complete",
):
    print(f"{key:28s} {bb84[key]}")


## Complete E91 workflow on B-D

B and D publicly identify coincidences without disclosing key outcomes, reserve four setting combinations for CHSH, correct the singlet anticorrelation, sample QBER, reconcile with Cascade, verify, and amplify privacy. CHSH outcomes never enter the key.


In [ ]:
e91 = result["e91"]
print("CHSH categories:")
for category in e91["bell_counts"]:
    print(
        f"  {category}: n={e91['bell_counts'][category]:4d}, "
        f"E={e91['bell_correlations'][category]: .4f}"
    )
print(f"Observed |S|: {abs(e91['observed_s']):.4f}")
print(f"Conservative S lower: {e91['s_lower']:.4f}")
for key in (
    "coincident_successful_detections", "key_rounds",
    "estimated_qber", "cascade_parity_requests",
    "cascade_corrections", "verification_accepted",
    "final_key_length", "final_keys_equal", "protocol_complete",
):
    print(f"{key:34s} {e91[key]}")
print("Privacy budget:", e91["privacy_budget"])


## Acceptance checks

A successful demonstration requires independent B-side report routing and equal non-empty final keys on both links.


In [ ]:
assert result["report_isolation"]["b_bb84_reports_only_from_bb84_detector"]
assert result["report_isolation"]["b_e91_reports_only_from_e91_detector"]
assert bb84["final_key_length"] > 0 and bb84["final_keys_equal"]
assert e91["final_key_length"] > 0 and e91["final_keys_equal"]
assert result["protocols_complete"]
print("Acceptance checks passed: both final-key pairs match.")


## Exercise: customize one physical arm

Predict what happens to E91 coincidences and final-key length if C-B is increased from 10 km to 20 km and its depolarizing probability is increased. Build the configuration below, then optionally run it by uncommenting the final line.


In [ ]:
from dataclasses import replace

custom_config = replace(
    config,
    master_seed=2030,
    e91_c_to_b=replace(
        config.e91_c_to_b,
        length_m=20_000,
        depolarizing_probability=0.05,
    ),
)
print(custom_config.e91_c_to_b)
# custom_result = run_four_node_qkd_trial(custom_config)


### Exercise answer scaffold

Longer fiber increases attenuation, so B should detect fewer E91 members and B-D coincidences should fall. More depolarization should reduce CHSH contrast and increase key disagreements. Depending on the seeded sample, the result will have a shorter final key or abort at the Bell, QBER, or minimum-key check.


## Pitfalls, security boundary, and extensions

- Do not connect both B quantum fibers to one detector input; separate physical receivers keep protocols isolated.
- Public channels are reliable and authenticated but still delayed. Lossy post-processing needs retransmission and timeout state machines, which are not included here.
- The E91 extraction length is a teaching budget using fixed CHSH/QBER margins and actual reconciliation leakage. Coincidence postselection means this is not a loophole-free or composable finite-key proof.
- Useful extensions include biased E91 basis probabilities, classical retransmission, finite-statistics confidence intervals, memories, or routed multi-hop public transport.
